%md
# 06 · Gold alerts
Applies label-free detection rules to gold account features and produces a scored alert queue for the agent. Thresholds come from the data's own distribution.

In [0]:
from pyspark.sql import functions as F

GOLD = "workspace.aml_gold"

accounts = spark.table(f"{GOLD}.account_features").filter(~F.col("is_hub"))

print(f"Accounts in scope: {accounts.count():,}")

In [0]:
q = accounts.approxQuantile(
    ["out_degree", "in_degree", "out_max_txns_per_day", "in_max_txns_per_day"],
    [0.99],
    0.001,
)

FAN_OUT_MIN = q[0][0]
FAN_IN_MIN = q[1][0]
VELOCITY_OUT_MIN = q[2][0]
VELOCITY_IN_MIN = q[3][0]
PASS_THROUGH_MIN = 0.9
MATERIAL_USD = 10_000

print(f"Fan-out:   out_degree >= {FAN_OUT_MIN}")
print(f"Fan-in:    in_degree >= {FAN_IN_MIN}")
print(f"Velocity:  out per day >= {VELOCITY_OUT_MIN} or in per day >= {VELOCITY_IN_MIN}")
print(f"Pass-through: ratio >= {PASS_THROUGH_MIN} and in_amount_usd >= {MATERIAL_USD:,}")

In [0]:
rule_cols = ["r_fan_out", "r_fan_in", "r_pass_through", "r_cycle", "r_velocity"]

flagged = (
    accounts
    .withColumn("r_fan_out", F.col("out_degree") >= FAN_OUT_MIN)
    .withColumn("r_fan_in", F.col("in_degree") >= FAN_IN_MIN)
    .withColumn(
        "r_pass_through",
        (F.col("pass_through_ratio") >= PASS_THROUGH_MIN) & (F.col("in_amount_usd") >= MATERIAL_USD),
    )
    .withColumn("r_cycle", F.col("in_cycle_2") | F.col("in_cycle_3"))
    .withColumn(
        "r_velocity",
        (F.col("out_max_txns_per_day") >= VELOCITY_OUT_MIN) | (F.col("in_max_txns_per_day") >= VELOCITY_IN_MIN),
    )
    .withColumn("rules_fired", sum(F.col(c).cast("int") for c in rule_cols))
)

display(flagged.select([F.sum(F.col(c).cast("int")).alias(c) for c in rule_cols]))

display(
    flagged.groupBy("rules_fired")
    .count()
    .orderBy("rules_fired")
)

In [0]:
rules = spark.createDataFrame(
    [
        ("r_fan_out", "Fan-out", f"Sent to at least {FAN_OUT_MIN:.0f} different accounts (top 1%)"),
        ("r_fan_in", "Fan-in", f"Received from at least {FAN_IN_MIN:.0f} different accounts (top 1%)"),
        ("r_pass_through", "Pass-through", f"At least {PASS_THROUGH_MIN:.0%} of incoming money sent back out, with at least {MATERIAL_USD:,} USD received"),
        ("r_cycle", "Cycle", "Part of a 2-hop or 3-hop loop where money returns to its origin"),
        ("r_velocity", "Velocity", f"At least {VELOCITY_OUT_MIN:.0f} outgoing or {VELOCITY_IN_MIN:.0f} incoming transactions in one day (top 1%)"),
    ],
    ["rule_id", "rule_name", "description"],
)

display(rules)

(
    rules.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD}.alert_rules")
)

In [0]:
rule_names = {
    "r_fan_out": "Fan-out",
    "r_fan_in": "Fan-in",
    "r_pass_through": "Pass-through",
    "r_cycle": "Cycle",
    "r_velocity": "Velocity",
}

alerts = (
    flagged
    .filter(F.col("rules_fired") >= 2)
    .withColumn("alert_id", F.concat(F.lit("ALR-"), F.substring(F.sha2("account_id", 256), 1, 10)))
    .withColumn(
        "triggered_rules",
        F.array_compact(F.array(*[F.when(F.col(c), F.lit(name)) for c, name in rule_names.items()])),
    )
    .withColumn("priority", F.when(F.col("rules_fired") >= 3, "HIGH").otherwise("MEDIUM"))
    .select(
        "alert_id",
        "account_id",
        "entity_id",
        "bank_id",
        "priority",
        "rules_fired",
        "triggered_rules",
        "out_degree",
        "in_degree",
        "out_amount_usd",
        "in_amount_usd",
        "pass_through_ratio",
        "in_cycle_2",
        "in_cycle_3",
        "out_max_txns_per_day",
        "in_max_txns_per_day",
        "first_seen_at",
        "last_seen_at",
        F.current_timestamp().alias("created_at"),
    )
)

display(alerts.orderBy(F.desc("rules_fired"), F.desc("in_amount_usd")).limit(10))

In [0]:
expected = flagged.filter(F.col("rules_fired") >= 2).count()
rows = alerts.count()
unique_ids = alerts.select("alert_id").distinct().count()
by_priority = {r["priority"]: r["count"] for r in alerts.groupBy("priority").count().collect()}
forbidden = [c for c in alerts.columns if "laundering" in c.lower() or "pattern" in c.lower()]

print(f"Alerts:       {rows:,}")
print(f"Unique IDs:   {unique_ids:,}")
print(f"By priority:  {by_priority}")

assert rows == expected, "Alert count does not match accounts with 2+ rules"
assert unique_ids == rows, "alert_id is not unique"
assert not forbidden, f"Label columns found in alerts: {forbidden}"

print("Quality gate passed")

(
    alerts.write
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{GOLD}.alerts")
)